# JobMatch AI — Notebook 04: Fine-tuning do BERTimbau

**Projeto:** JobMatch AI — Sistema de Matching Currículo-Vaga com NLP/Deep Learning
**Autor:** Eduardo Matos
**Etapa:** Fine-tuning supervisionado do BERTimbau para prever fit_percentual

## O que é fine-tuning, e por que chegamos até aqui só agora

Fine-tuning é o processo de pegar um modelo já pré-treinado (BERTimbau, que
"aprendeu português" com um corpus gigante) e continuar treinando ele numa
tarefa específica — no nosso caso, prever o quão bem uma vaga combina com o
perfil do autor (fit_percentual, de 0 a 100).

Diferente dos notebooks anteriores (que usavam o BERT "congelado", só extraindo
embeddings prontos), aqui os pesos internos do modelo são ajustados durante o
treinamento — é uma técnica mais poderosa, mas também mais exigente em dados.

## Por que só fizemos isso agora

Fine-tuning de transformers geralmente precisa de 100+ exemplos para começar a
generalizar razoavelmente. Com as 15 vagas manuais originais, isso era inviável.
Com o dataset ampliado para 162 vagas (15 manuais + 147 via API), estamos no
limiar mínimo para tentar essa abordagem — ainda um volume modesto para deep
learning, mas suficiente para uma prova de conceito metodologicamente correta.

## Enquadramento do problema

Tratamos isso como um problema de **regressão**: o modelo recebe o texto da
vaga e aprende a prever diretamente um número entre 0 e 100 (fit_percentual),
em vez de classificar em categorias discretas. Isso preserva a granularidade
do rótulo original.

## Limitação assumida desde já

Com N=162 (e um split de treino/validação reduzindo ainda mais os dados de
treino efetivos), não esperamos um modelo com alta capacidade de generalização
— o objetivo aqui é demonstrar o pipeline completo e correto de fine-tuning,
com avaliação honesta dos resultados, não produzir um modelo pronto para
produção.

## Etapas deste notebook
1. Preparar os dados (split treino/validação, tokenização em lote)
2. Carregar o BERTimbau com uma "cabeça" de regressão
3. Configurar e executar o treinamento
4. Avaliar o modelo (métricas de erro)
5. Comparar com o baseline de similaridade (Notebook 03)
6. Salvar o modelo treinado

In [ ]:
!pip install transformers datasets accelerate -q


## 1. Verificando o ambiente de execução

Fine-tuning de transformers é computacionalmente pesado — rodar em CPU pode
levar horas mesmo com um dataset pequeno. Antes de tudo, confirmamos se o
Colab está usando GPU nesta sessão.

In [ ]:
import torch

print("GPU disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Dispositivo:", torch.cuda.get_device_name(0))
else:
    print("ATENÇÃO: rodando em CPU. O treinamento será bem mais lento.")

GPU disponível: True
Dispositivo: Tesla T4


## 2. Preparando os dados para o fine-tuning

Antes de treinar, precisamos:
1. Dividir o dataset em treino e validação (para avaliar o modelo em dados
   que ele não viu durante o treino)
2. Tokenizar todas as descrições em lote, no formato que o modelo espera
3. Normalizar o fit_percentual (0-100) para uma escala 0-1, prática comum
   que ajuda a estabilidade do treinamento

**Decisão sobre o split:** com apenas 162 exemplos, reservamos 80% para
treino (130 vagas) e 20% para validação (32 vagas) — proporção padrão,
mesmo sabendo que 32 exemplos de validação é uma amostra pequena para uma
avaliação estatisticamente robusta. Usamos stratify por faixa de fit para
garantir que treino e validação tenham representação proporcional das
categorias (alto/médio/baixo).

In [ ]:
import pandas as pd

CAMINHO_CSV = '/content/drive/MyDrive/jobmatch-ai/data/vagas_clean_v2.csv'

from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv(CAMINHO_CSV)
print('Shape:', df.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shape: (162, 13)


In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

# # Criar faixas de fit para estratificação (garante proporção similar nos dois splits)
df['faixa_fit'] = pd.cut(df['fit_percentual'], bins=[-1, 29, 59, 100], labels=['baixo', 'medio', 'alto'])

df_treino, df_val = train_test_split(df, test_size=0.2, random_state=42, stratify=df['faixa_fit'])

print(f'Treino: {len(df_treino)} vagas')
print(f'Validação: {len(df_val)} vagas')

print("\n=== distribuição de faixas no treino ===")
print(df_treino['faixa_fit'].value_counts())

print("\n=== distribuição de faixas na validação ===")
print(df_val['faixa_fit'].value_counts())


Treino: 129 vagas
Validação: 33 vagas

=== distribuição de faixas no treino ===
faixa_fit
baixo    49
alto     49
medio    31
Name: count, dtype: int64

=== distribuição de faixas na validação ===
faixa_fit
baixo    13
alto     12
medio     8
Name: count, dtype: int64


## 3. Tokenizando e montando o Dataset

Convertemos os dataframes de treino e validação para o formato `Dataset` da
biblioteca Hugging Face `datasets` — isso é necessário porque o `Trainer`
(ferramenta de treinamento que usaremos a seguir) espera os dados nesse
formato específico, com tokenização já aplicada e um tensor de "labels"
correspondente ao fit_percentual normalizado.

Usamos `max_length=256` (decisão tomada no Notebook 02, com folga sobre o
máximo observado de 231 tokens), com padding e truncation habilitados para
garantir que todos os exemplos de um mesmo lote (batch) tenham o mesmo tamanho.

In [ ]:
from transformers import AutoTokenizer

NOME_MODELO = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(NOME_MODELO)

print("Tokenizer carregado com sucesso!")

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer carregado com sucesso!


In [ ]:
from datasets import Dataset

# Normalizar fit_percentual para escala 0-1 (boa prática para regressão com redes neurais)
df_treino = df_treino.copy()
df_val = df_val.copy()
df_treino['label'] = df_treino['fit_percentual'] / 100.0
df_val['label'] = df_val['fit_percentual'] / 100.0

# Converter para Dataset do Hugging Face
dataset_treino = Dataset.from_pandas(df_treino[['descricao', 'label']].reset_index(drop=True))
dataset_val = Dataset.from_pandas(df_val[['descricao', 'label']].reset_index(drop=True))

def tokenizar_funcao(exemplos):
    return tokenizer(exemplos['descricao'], padding='max_length', truncation=True, max_length=256)

dataset_treino_tok = dataset_treino.map(tokenizar_funcao, batched=True)
dataset_val_tok = dataset_val.map(tokenizar_funcao, batched=True)

print("Exemplo tokenizado (treino):")
print(dataset_treino_tok[0].keys())
print("\nTamanho do dataset de treino:", len(dataset_treino_tok))
print("Tamanho do dataset de validação:", len(dataset_val_tok))


Map:   0%|          | 0/129 [00:00<?, ? examples/s]

Map:   0%|          | 0/33 [00:00<?, ? examples/s]

Exemplo tokenizado (treino):
dict_keys(['descricao', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])

Tamanho do dataset de treino: 129
Tamanho do dataset de validação: 33


## 4. Carregando o BERTimbau com cabeça de regressão

Para fine-tuning em tarefa de regressão, usamos a classe
`AutoModelForSequenceClassification` configurada com `num_labels=1` — isso
adiciona uma camada final ao BERTimbau que produz um único valor numérico
contínuo (em vez de probabilidades de classes discretas), que é o que
precisamos para prever o fit_percentual.

In [ ]:
from transformers import AutoModelForSequenceClassification

modelo = AutoModelForSequenceClassification.from_pretrained(
    NOME_MODELO,
    num_labels=1,
    problem_type="regression"
)

modelo.to("cuda")  # move o modelo para a GPU

print("Modelo carregado e movido para GPU com sucesso!")
print("Número de parâmetros:", sum(p.numel() for p in modelo.parameters()))

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Modelo carregado e movido para GPU com sucesso!
Número de parâmetros: 108923905


## 5. Configurando os argumentos de treinamento

Definimos os hiperparâmetros do fine-tuning. Com apenas 129 exemplos de treino,
algumas escolhas são deliberadamente conservadoras para reduzir o risco de
overfitting severo:

- **Poucas épocas** (3-4): mais que isso, com tão poucos dados, tende a fazer
  o modelo simplesmente memorizar os exemplos de treino
- **Learning rate baixo** (2e-5): padrão para fine-tuning de BERT, evita que
  o modelo "esqueça" o conhecimento de português já aprendido no pré-treinamento
- **Batch size pequeno** (8): adequado ao volume de dados e à memória da GPU
  gratuita do Colab
- **Avaliação a cada época**: para acompanhar se o modelo está de fato
  aprendendo ou apenas decorando o treino (sinal de overfitting)

In [ ]:
from transformers import TrainingArguments

argumentos_treino = TrainingArguments(
    output_dir="/content/resultados_treino",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_dir="/content/logs",
    logging_steps=10,
    report_to="none",
)

print("Argumentos de treinamento configurados.")

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Argumentos de treinamento configurados.


## 6. Montando o Trainer e definindo a métrica de avaliação

O `Trainer` da Hugging Face encapsula todo o loop de treinamento (forward pass,
backpropagation, otimização, avaliação) — não precisamos escrever isso manualmente.

Definimos uma função de métrica customizada para acompanhar o erro do modelo
em termos interpretáveis: MAE (Mean Absolute Error, erro médio absoluto) na
escala original de 0-100, em vez de só a "loss" bruta do treinamento.

In [ ]:
from transformers import Trainer
import numpy as np

def calcular_metricas(eval_pred):
    predicoes, labels = eval_pred
    predicoes = predicoes.flatten()

    # Convertendo de volta para escala 0-100 (estava normalizado 0-1)
    predicoes_escala = predicoes * 100
    labels_escala = labels * 100

    mae = np.mean(np.abs(predicoes_escala - labels_escala))
    rmse = np.sqrt(np.mean((predicoes_escala - labels_escala) ** 2))

    return {"mae": mae, "rmse": rmse}

trainer = Trainer(
    model=modelo,
    args=argumentos_treino,
    train_dataset=dataset_treino_tok,
    eval_dataset=dataset_val_tok,
    compute_metrics=calcular_metricas,
)

print("Trainer configurado. Pronto para iniciar o treinamento.")

Trainer configurado. Pronto para iniciar o treinamento.


## 7. Executando o fine-tuning

Esta célula executa o treinamento propriamente dito: o modelo verá as 129
vagas de treino repetidamente (4 épocas), ajustando seus pesos internos a
cada lote para minimizar o erro entre o fit_percentual previsto e o real.

A cada época, o modelo é avaliado no conjunto de validação (33 vagas que ele
nunca viu durante o treino) — isso permite acompanhar se o modelo está
generalizando ou apenas memorizando os exemplos de treino (overfitting).

**Expectativa realista:** com apenas 129 exemplos de treino, é esperado que
o modelo apresente sinais de overfitting nas últimas épocas (loss de treino
caindo, mas loss de validação estagnando ou subindo). Isso será documentado
como parte da análise, não escondido.

In [ ]:
resultado_treino = trainer.train()

print("\n=== Treinamento concluído ===")
print(resultado_treino)

Epoch,Training Loss,Validation Loss,Mae,Rmse
1,0.117403,0.048771,18.446014,22.084055
2,0.039833,0.036588,16.570063,19.128119
3,0.027743,0.027176,12.454389,16.485294
4,0.018564,0.024385,12.372149,15.615739


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== Treinamento concluído ===
TrainOutput(global_step=68, training_loss=0.043474521268816554, metrics={'train_runtime': 60.5438, 'train_samples_per_second': 8.523, 'train_steps_per_second': 1.123, 'total_flos': 67882042791936.0, 'train_loss': 0.043474521268816554, 'epoch': 4.0})


## 8. Diagnóstico de erros — onde o modelo mais erra

Analisamos individualmente as previsões no conjunto de validação, identificando
os casos de maior erro. Isso é mais informativo que só olhar a média (MAE/RMSE),
porque revela **padrões** nos erros — por exemplo, se o modelo erra
sistematicamente num tipo específico de vaga, isso aponta uma limitação
concreta a documentar.

In [ ]:
import numpy as np

# Gerar previsões no conjunto de validação
predicoes_output = trainer.predict(dataset_val_tok)
predicoes = predicoes_output.predictions.flatten() * 100  # volta pra escala 0-100
reais = df_val['fit_percentual'].values

# Montar tabela de comparação
df_erros = pd.DataFrame({
    'titulo': df_val['titulo'].values,
    'area': df_val['area'].values,
    'senioridade': df_val['senioridade'].values,
    'fit_real': reais,
    'fit_previsto': predicoes.round(1),
})
df_erros['erro_absoluto'] = (df_erros['fit_real'] - df_erros['fit_previsto']).abs().round(1)

df_erros_ordenado = df_erros.sort_values('erro_absoluto', ascending=False)

print("=== Top 10 maiores erros ===")
print(df_erros_ordenado.head(10).to_string())

print("\n=== Top 5 melhores acertos ===")
print(df_erros_ordenado.tail(5).to_string())

print(f"\n=== MAE recalculado (conferência): {df_erros['erro_absoluto'].mean():.2f} ===")

=== Top 10 maiores erros ===
                                                          titulo                 area       senioridade  fit_real  fit_previsto  erro_absoluto
4                                      Desenvolvedor Python - RJ      Desenvolvimento            junior        19     57.400002           38.4
19                                        Analista Suporte Ti Pl         Fora de Area  Não identificado        15     50.700001           35.7
13  Principal Service Engineer - IA Engineer | Remoto | Prezensa  Engenharia de Dados            Sênior        25     50.900002           25.9
24                                            Cientista de Dados     Ciencia de Dados            Sênior        43     67.199997           24.2
12                                     Cientista de Dados Júnior     Ciencia de Dados            Júnior        80     56.900002           23.1
6              Analista De Inteligencia Artificial (Ia) - Junior     Ciencia de Dados            Júnior        75

## 9. Validando a comparação com um baseline ingênuo

Antes de afirmar que o modelo "aprendeu" algo de fato, é preciso confirmar
que ele supera uma previsão trivial: simplesmente prever a média do
fit_percentual de treino para toda vaga nova, ignorando o texto completamente.
Se o modelo fine-tuned não superar esse baseline por uma margem clara, o
fine-tuning não teria valor prático.

In [ ]:
# Baseline ingênuo: sempre prever a média do TREINO (nunca usar estatística de validação/teste)
media_treino = df_treino['fit_percentual'].mean()
print(f"Média do fit_percentual no treino: {media_treino:.2f}")

# Aplicar essa previsão constante em todo o conjunto de validação
previsao_baseline = np.full(len(df_val), media_treino)
mae_baseline = np.mean(np.abs(df_val['fit_percentual'].values - previsao_baseline))
rmse_baseline = np.sqrt(np.mean((df_val['fit_percentual'].values - previsao_baseline) ** 2))

print(f"\n=== Baseline ingênuo (sempre prever a média) ===")
print(f"MAE: {mae_baseline:.2f}")
print(f"RMSE: {rmse_baseline:.2f}")

print(f"\n=== Modelo fine-tuned (BERTimbau) ===")
print(f"MAE: {df_erros['erro_absoluto'].mean():.2f}")
print(f"RMSE: {np.sqrt(np.mean((df_erros['fit_real'] - df_erros['fit_previsto'])**2)):.2f}")

melhoria_mae = (1 - (df_erros['erro_absoluto'].mean() / mae_baseline)) * 100
print(f"\n=== Melhoria relativa do modelo sobre o baseline ===")
print(f"MAE: {melhoria_mae:.1f}% menor que o baseline ingênuo")

Média do fit_percentual no treino: 43.21

=== Baseline ingênuo (sempre prever a média) ===
MAE: 25.22
RMSE: 27.62

=== Modelo fine-tuned (BERTimbau) ===
MAE: 12.36
RMSE: 15.61

=== Melhoria relativa do modelo sobre o baseline ===
MAE: 51.0% menor que o baseline ingênuo


## Conclusões — Fine-tuning do BERTimbau (N=162)

**Métricas finais (conjunto de validação, N=33):**
- MAE: 12.36 pontos (escala 0-100) — 51.0% menor que o baseline ingênuo (25.22)
- RMSE: 15.61 pontos — 43.5% menor que o baseline ingênuo (27.62)
- Validation loss caiu consistentemente ao longo das 4 épocas, sem sinais
  claros de overfitting

**Padrão de erro identificado — regressão à média:** o modelo tende a
subestimar vagas de fit muito alto (>75) e superestimar vagas de fit muito
baixo (<20), "puxando" as previsões em direção ao centro da distribuição —
esperado com volume de treino modesto (129 exemplos).

**Limitação recorrente confirmada:** os maiores erros concentram-se em vagas
com senioridade Sênior/Pleno, reforçando o padrão já observado nos notebooks
de embeddings — o modelo tem dificuldade em capturar o impacto da senioridade
no fit, mesmo após fine-tuning supervisionado.

**Validação contra baseline:** o fine-tuning reduziu o erro médio em 51%
comparado a simplesmente prever a média do dataset, confirmando que o modelo
aprendeu sinal real do texto das vagas, não apenas a distribuição geral dos
rótulos.

**Trabalho futuro:** ampliar o dataset (especialmente exemplos Sênior/Pleno
rotulados corretamente) tende a reduzir tanto a regressão à média quanto a
confusão de senioridade.

In [ ]:
CAMINHO_MODELO = '/content/drive/MyDrive/jobmatch-ai/models/bertimbau_fit_v1'
trainer.save_model(CAMINHO_MODELO)
tokenizer.save_pretrained(CAMINHO_MODELO)

print(f"Modelo salvo em: {CAMINHO_MODELO}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo salvo em: /content/drive/MyDrive/jobmatch-ai/models/bertimbau_fit_v1
